# Pré-processamento e Engenharia Espacial (H3)
Objetivo: Executar a etapa de pré-processamento dos microdados (SSP-SP), englobando a filtragem de anomalias espaciais, a aplicação do sistema de indexação Uber H3 e a agregação espaço-temporal para estruturação do painel longitudinal.


In [1]:
import pandas as pd
import numpy as np
import h3
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

ANO_BASE = "2023"
H3_RES = 9  # aprox 174m de aresta


## 1. Carregamento e Limpeza
Leitura e concatenação dos arquivos tabulares do exercício correspondente.


In [2]:

files_furto = glob.glob(f'../data/SP/raw/FURTO_VEICULO_{ANO_BASE}_*.xls')
files_roubo = glob.glob(f'../data/SP/raw/ROUBO_VEICULO_{ANO_BASE}_*.xls')
all_files = files_furto + files_roubo

print(f"Total de arquivos encontrados para {ANO_BASE}: {len(all_files)}")

dfs = []
for f in all_files:
    try:
        df_temp = pd.read_csv(f, sep='\t', encoding='utf-16le', dtype=str)
        df_temp['CRIME'] = 'FURTO' if 'FURTO' in f else 'ROUBO'
        dfs.append(df_temp)
    except Exception as e:
        print(f"Erro no arquivo {f}: {e}")

df = pd.concat(dfs, ignore_index=True)
print(f"Total de registros brutos: {df.shape[0]}")


Total de arquivos encontrados para 2023: 24
Erro no arquivo ../data/SP/raw/FURTO_VEICULO_2023_02.xls: Error tokenizing data. C error: EOF inside string starting at row 11844
Total de registros brutos: 331996


### 1.1 Limpeza das Coordenadas Geográficas


In [7]:
# Ignora coordenadas nulas e converte para numérico
df_geo = df.dropna(subset=['LATITUDE', 'LONGITUDE']).copy()
df_geo['LATITUDE'] = pd.to_numeric(df_geo['LATITUDE'].str.replace(',', '.'), errors='coerce')
df_geo['LONGITUDE'] = pd.to_numeric(df_geo['LONGITUDE'].str.replace(',', '.'), errors='coerce')

# Remove nulos pós-conversão e outliers
df_geo = df_geo.dropna(subset=['LATITUDE', 'LONGITUDE'])
df_geo = df_geo[(df_geo['LATITUDE'] != 0) & (df_geo['LONGITUDE'] != 0)]
df_geo = df_geo[(df_geo['LATITUDE'] < -19) & (df_geo['LATITUDE'] > -26)]
df_geo = df_geo[(df_geo['LONGITUDE'] < -44) & (df_geo['LONGITUDE'] > -54)]

print(f"Total de registros válidos com coordenadas: {df_geo.shape[0]}")


Total de registros válidos com coordenadas: 294385


### 1.2 Limpeza Temporal


In [8]:

df_geo['DATAOCORRENCIA'] = pd.to_datetime(df_geo['DATAOCORRENCIA'], errors='coerce', dayfirst=True)
df_geo = df_geo.dropna(subset=['DATAOCORRENCIA'])

df_geo['ANO_SEMANA'] = df_geo['DATAOCORRENCIA'].dt.strftime('%Y-%U')

print("Frequência semanal de ocorrências:")
display(df_geo['ANO_SEMANA'].value_counts().sort_index().head())


Frequência semanal de ocorrências:


ANO_SEMANA
2003-44    1
2006-23    1
2006-40    1
2007-13    1
2007-21    1
Name: count, dtype: int64

## 2. Engenharia Espacial: Indexação H3
Conversão das coordenadas contínuas (Latitude, Longitude) em polígonos discretos padronizados, utilizando o algoritmo H3 na Resolução definida.


In [9]:
def get_h3(row):
    return h3.latlng_to_cell(row['LATITUDE'], row['LONGITUDE'], H3_RES)

df_geo['H3_INDEX'] = df_geo.apply(get_h3, axis=1)
print(f"Total de hexágonos únicos atingidos: {df_geo['H3_INDEX'].nunique()}")


Total de hexágonos únicos atingidos: 30977


## 3. Agregação Semanal (Painel Longitudinal)
Agrupamento espacial-temporal de incidentes, resultando na criação de uma variável binária alvo (Target) indicativa de ocorrência criminal.


In [10]:

df_agg = df_geo.groupby(['H3_INDEX', 'ANO_SEMANA']).size().reset_index(name='QTD_CRIMES')

df_agg['ALVO'] = (df_agg['QTD_CRIMES'] > 0).astype(int)

os.makedirs('../data/SP/processed', exist_ok=True)
df_agg.to_csv('../data/SP/processed/painel_h3_semanal.csv', index=False)

print("Painel processado e salvo!")
display(df_agg.head())


Painel processado e salvo!


,H3_INDEX,ANO_SEMANA,QTD_CRIMES,ALVO
0,89a80049a63ffff,2023-43,6,1
1,89a800519d3ffff,2023-04,4,1
2,89a80051d03ffff,2023-23,1,1
3,89a80051d03ffff,2023-44,2,1
4,89a80051d17ffff,2023-16,2,1
